# 02b · E1 error analysis (exploratory)

Why do models get items right in English but wrong in Hausa? This notebook draws 50 such **discordant** items (correct in `en_p0`, wrong in `ha_p0`): 25 from Llama-3.2-3B and 25 from MedGemma-4B, with the fixed seed in `src/config.py` (deviation 4). You code each one into one of the four pre-registered categories. No hypothesis test is attached.

- **Part A** draws the sample and exports a coding sheet. The sheet hides which model gave each answer, and rows are shuffled.
- **You** code the sheet.
- **Part B** checks the sheet, joins the models back on, and writes `e1_summary.json`.

No GPU needed; a CPU runtime is fine.

## 0 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_DIR = "/content/hausa-med-qa"
import os
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/avvas200/hausa-med-qa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log -1 --format="%h %s"

In [ ]:
import json, shutil
import pandas as pd
from src import config as C, data as D, e1 as E1

manifest = json.load(open("results/data_manifest.json"))
for f in ["eval_en.jsonl", "eval_ha.jsonl", "segment_status.json"]:
    assert D.sha256(C.DATA / f) == manifest["sha256"][f], f"{f} differs from the frozen manifest"
eval_en = D.read_jsonl(C.DATA / "eval_en.jsonl")
eval_ha = D.read_jsonl(C.DATA / "eval_ha.jsonl")
seg_status = json.load(open(C.DATA / "segment_status.json"))["eval"]
ids = [r["id"] for r in eval_en]
print(len(ids), "items; data files match the manifest")

# Part A · Draw the sample and export the sheet

In [ ]:
rows, pools = E1.draw_sample(ids)
print("discordant pool sizes:", pools)
key = pd.DataFrame(rows)
print("items drawn for both models:", int(key.groupby("id").model.nunique().eq(2).sum()))

sheet = E1.coding_sheet(rows, eval_en, eval_ha, seg_status)
sheet_path, key_path = C.DATA / "e1_coding.csv", C.DATA / "e1_key.csv"
if sheet_path.exists():
    # The draw is seeded, so a re-run must reproduce the exported sheet exactly.
    old = pd.read_csv(sheet_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    assert old.equals(sheet.astype(str)), "Re-drawn sample differs from the exported sheet - stop and check"
    print("sheet already exported; re-draw matches it, nothing overwritten")
else:
    sheet.to_csv(sheet_path, index=False, encoding="utf-8-sig")
    key.to_csv(key_path, index=False, encoding="utf-8-sig")
    print(len(sheet), "rows ->", sheet_path)

### How to code the sheet

Open `e1_coding.csv` in Google Sheets. Each row shows the English and Hausa question and options (canonical order), the correct letter (`gold`), and the letter the model chose in Hausa (`model_answer_ha`). The `*_status` columns say how each Hausa segment was produced: `auto` (machine translation, not reviewed), `reviewed_kept`, `reviewed_edited`, `english` (kept in English after your review), `code_or_number`.

Go through the checks **in this order** and use the **first** code that applies:

| order | `code` | use when | `segment` |
|---|---|---|---|
| 1 | `format` | the model's output is not a usable answer (`model_answer_ha` = none) | leave empty |
| 2 | `term` | a **medical term** in the Hausa is wrong, invented, or not recognisable to a Hausa speaker (an English term kept on purpose is **not** an error) | where it is: `stem`, `A`, `B`, `C` or `D` |
| 3 | `translation` | other meaning change in the Hausa: wrong or missing word, lost negation ("not", "except"), wrong number, garbled sentence | where it is: `stem`, `A`–`D` |
| 4 | `reasoning` | the Hausa says the same as the English, and the model still chose wrongly | leave empty |

Tips:
- Judge the Hausa **as a Hausa reader would**, against the English. Ask: would a Hausa-speaking medical student get this right from the Hausa text alone?
- Only code an error if it could plausibly change the answer. A slightly awkward but clear Hausa phrase is `reasoning`, not `translation`.
- If there are several errors, code the one that most likely caused the wrong answer, and mention the others in `notes`.
- With the prefill, Llama and MedGemma parsed 100% of Hausa answers, so `format` is expected to be very rare. It stays in the scheme because it was pre-registered.
- Don't open `e1_key.csv` (it shows which model is which) until you've finished.

When you're done, download it as CSV and save it to Drive as `hausa-med-qa/data/e1_coding_done.csv`.

# Part B · Check and summarise

In [ ]:
coded = E1.read_codes(C.DATA / "e1_coding_done.csv", C.DATA / "e1_key.csv")
print(pd.crosstab(coded.model, coded.code, margins=True).to_string())

In [ ]:
summary = E1.summarise(coded, eval_en, seg_status)
summary["sample"] = {"models": C.E1_MODELS, "n_per_model": C.E1_N_PER_MODEL, "seed": C.E1_SEED,
                     "discordant_pool_sizes": pools}
summary["sha256"] = {f: D.sha256(C.DATA / f) for f in ["e1_coding_done.csv", "e1_key.csv"]}

for m, s in summary["per_model"].items():
    print(f"\n== {m} (n={s['n']}) ==")
    for c, v in s["codes"].items():
        print(f"  {c:12s} {v['n']:3d}  {v['prop']:.2f}  95% CI {v['ci95'][0]:.2f}-{v['ci95'][1]:.2f}")
    print("  text errors are in:", s["text_error_location"])
    print("  status of those segments:", s["text_error_segment_status"])

json.dump(summary, open(C.RESULTS / "e1_summary.json", "w"), indent=2)
for f in ["e1_coding_done.csv", "e1_key.csv"]:
    shutil.copy(C.DATA / f, C.RESULTS / f)
print("\nsaved to", C.RESULTS)

## Next
Paste the Part B output into the chat. Then copy `e1_summary.json`, `e1_coding_done.csv` and `e1_key.csv` from Drive `hausa-med-qa/results/` into the repo's `results/` folder and commit them.